In [0]:
dbutils.widgets.removeAll()


In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F


In [0]:

dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_music_silver = spark.table(f"{catalogo}.{esquema_source}.music_trends_transformed")
df_reviews_silver = spark.table(f"{catalogo}.{esquema_source}.spotify_reviews_clean")

In [0]:
df_music_silver = spark.table(f"{catalogo}.{esquema_source}.music_trends_transformed")

In [0]:
df_gold_music = df_music_silver.groupBy(col("itunes_genre")).agg(
    count(col("track_id")).alias("total_tracks"),
    round(avg(col("popularity")), 2).alias("avg_popularity"),
    max(col("price_usd")).alias("max_price_usd"),
    sum(when(col("is_hit") != "Regular", 1).otherwise(0)).alias("total_hits")
).orderBy(col("total_tracks").desc())


In [0]:
df_gold_music.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.gold_music_metrics")

In [0]:
df_reviews_silver = spark.table(f"{catalogo}.{esquema_source}.spotify_reviews_clean")

In [0]:
df_gold_reviews = df_reviews_silver.groupBy(col("sentiment_category")).agg(
    count(col("Review")).alias("total_reviews"),
    round(avg(col("Rating")), 2).alias("avg_rating"),
    sum(col("Total_thumbsup")).alias("total_thumbsup")
).orderBy(col("total_reviews").desc())

In [0]:
df_gold_reviews.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.gold_reviews_metrics")

In [0]:
df_gold_segments = df_music_silver.groupBy(
    col("itunes_genre"),
    col("popularity_category"),
    col("is_hit")
).agg(
    count(col("track_id")).alias("total_tracks"),
    round(avg(col("popularity")), 2).alias("avg_popularity"),
    round(avg(col("duration_ms")) / 60000, 2).alias("avg_duration_minutes")
).orderBy(col("total_tracks").desc())

df_gold_segments.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.gold_music_segments")

In [0]:
df_gold_segments.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.gold_music_segments")

print("Capa Golden consolidada y procesada con éxito.")

Capa Golden consolidada y procesada con éxito.
